# 01 — Análise da imagem: autocorrelação espacial, lags e amostragem

**Executar uma vez por imagem SAR** (altere `DATA`).

| Entrada (config) | Saída |
|---|---|
| imagem, ordem das bandas, **amostras** (vetor de polígonos **ou** raster de classes) | `01_lags_amostragem.xlsx`, `amostras.gpkg` (camadas `treino` e `validacao`), `amostras_raster.tif` |

**Método (Seção 4.3.1.1 da dissertação):**
1. Para cada polígono e polarização, calcula-se a autocorrelação espacial nas direções horizontal e vertical.
2. O lag de cada direção é o primeiro em que a correlação fica abaixo de 0,3.
3. Por classe, toma-se o maior lag entre as polarizações, desconsiderando outliers (critério IQR).
4. Gera-se a amostragem sistemática por esses lags.

Os lags seguem direto para a amostragem, associados às classes pelo **nome**.


## Parâmetros

In [ ]:
CONFIG = 'config_projeto_2010.json'
DATA = '2010'

ALVO_CORRELACAO = 0.3      # limiar de descorrelação
ESTIMADOR = 'pairs'        # 'pairs' (código original) | 'total' (Eq. 4.3)
INCLUSIVO = True          # False: R < limiar (código) | True: R <= limiar (texto)
TRANSFORMACAO = 'sqrt'     # 'sqrt' intensidade -> amplitude antes da autocorrelação (como no original); None se já for amplitude
USAR_IQR = True            # máximo sem outliers
ESTATISTICA_LAG = 'max'    # 'max' (dissertação) | 'median'
LAGS_MANUAIS = None        # ex.: {'AP': (4, 4), 'AC': (4, 4), ...} para impor uma tabela já publicada

PLOTAR_PERFIS = False      # um gráfico por polígono x banda (muitos!)
SALVAR_FIGURAS = False     # True: figuras_01.pdf

## Configuração e arquivos

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))      # pasta que contém sar_classification

import rasterio
from rasterio import features
from rasterio.enums import MergeAlg
from rasterio.warp import reproject, Resampling
from shapely.geometry import shape
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sar_classification import (autocorrelation as ac, sampling, samples as smp, fitting as fit,
                                separability as sep, models as mdl, validation as val, raster_io as rio,
                                transitions as trn, cmap, trajectory_assessment as ta, plots, reporting)

with open(CONFIG, encoding='utf-8') as f:
    cfg = json.load(f)
RESULTADOS = Path(cfg['raiz_resultados'])
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

img = cfg['imagens'][DATA]
CLASSES = cfg['classes']                         # ordem = códigos 1..K do raster de amostras
BANDAS = img['bandas']                           # ordem REAL das bandas no arquivo
OUT = RESULTADOS / DATA
OUT.mkdir(parents=True, exist_ok=True)
ARQ_PLANILHA, ARQ_AMOSTRAS = OUT / '01_lags_amostragem.xlsx', OUT / 'amostras.gpkg'
pdf = reporting.open_pdf(OUT / 'figuras_01.pdf') if SALVAR_FIGURAS else None
print(f'Imagem {DATA}: {img["imagem"]}\nBandas: {BANDAS}\nSaídas: {OUT}')

## 1. Amostras: uma única entrada, vetor ou raster

O config informa **um só arquivo** em `amostras`. O formato é reconhecido pela extensão, e o notebook gera a partir dele as duas representações usadas adiante, sempre na grade da imagem:

| Representação | Usada em | Vetor de polígonos (`.shp`, `.gpkg`, `.geojson`) | Raster de classes (`.tif`) |
|---|---|---|---|
| polígonos | seção 2 (autocorrelação por polígono) | lidos do arquivo | gerados: cada região contínua de uma classe (vizinhança 4) vira um polígono |
| raster de classes | seção 4 (grade de treino/validação) | gerado por rasterização (pixel entra se o **centro** está no polígono) | lido do arquivo; reamostrado por vizinho mais próximo se não estiver na grade da imagem |

O raster usado na amostragem é gravado em `amostras_raster.tif`, para conferência no QGIS.

**Campos do config (`imagens.<data>`):**
- `amostras`: caminho do arquivo;
- `coluna_classe` (só vetor, padrão `"Classe"`): coluna com o nome da classe;
- `nomes_classes` (só vetor, opcional): tradução dos nomes do arquivo para os de `classes`, ex.: `{"CL": "AC"}`.

No raster, os códigos devem ser 1..K na ordem de `classes`; 0, -9999 e o nodata do arquivo são tratados como ausência de amostra.

> **Raster como entrada:** polígonos vizinhos da mesma classe que se tocam viram um único polígono, e a autocorrelação é calculada sobre a região inteira. Isso muda o número de polígonos por classe e, portanto, o critério IQR da seção 3.

In [ ]:
ARQ_ENTRADA = img.get('amostras')
antigas = [k for k in ('poligonos_amostras', 'raster_amostras', 'coluna_classe_poligonos', 'nomes_poligonos') if k in img]
if ARQ_ENTRADA is None or antigas:
    raise KeyError(f'imagens.{DATA}: informe somente "amostras" (vetor OU raster), com "coluna_classe" e "nomes_classes" '
                   f'se for vetor. Chaves antigas a remover: {antigas or "nenhuma"}.')

EXT_VETOR = ('.shp', '.gpkg', '.geojson', '.json', '.sqlite')
ORIGEM = 'vetor' if Path(ARQ_ENTRADA).suffix.lower() in EXT_VETOR else 'raster'
COL = 'classe'                                   # coluna de classe usada daqui em diante
CODIGO = {c: i + 1 for i, c in enumerate(CLASSES)}
ARQ_RASTER_AMOSTRAS = OUT / 'amostras_raster.tif'

with rasterio.open(img['imagem']) as src:
    perfil = src.profile.copy()
    GRADE_TRANSFORM, GRADE_SHAPE, GRADE_CRS = src.transform, (src.height, src.width), src.crs

if ORIGEM == 'vetor':
    poligonos = gpd.read_file(ARQ_ENTRADA)
    col_arquivo = img.get('coluna_classe', 'Classe')
    if col_arquivo not in poligonos.columns:
        raise KeyError(f'Coluna "{col_arquivo}" não existe em {ARQ_ENTRADA}. Colunas: {list(poligonos.columns)}')
    if poligonos.crs is None:
        raise ValueError(f'{ARQ_ENTRADA} sem CRS definido (.prj ausente).')
    poligonos[COL] = poligonos[col_arquivo].astype(str).replace(img.get('nomes_classes') or {})
    fora = sorted(set(poligonos[COL]) - set(CLASSES))
    if fora:
        raise ValueError(f'Classes fora de cfg["classes"]: {fora} — ajuste "nomes_classes" no config.')
    poligonos = poligonos[[COL, 'geometry']].to_crs(GRADE_CRS).reset_index(drop=True)

    raster_classes = features.rasterize(((g, CODIGO[c]) for g, c in zip(poligonos.geometry, poligonos[COL])),
                                        out_shape=GRADE_SHAPE, transform=GRADE_TRANSFORM, fill=0, dtype='uint8')
    cobertura = features.rasterize(((g, 1) for g in poligonos.geometry), out_shape=GRADE_SHAPE,
                                   transform=GRADE_TRANSFORM, fill=0, dtype='uint16', merge_alg=MergeAlg.add)
    n_sobrepostos = int((cobertura > 1).sum())
    if n_sobrepostos:
        print(f'Aviso: {n_sobrepostos} pixels cobertos por mais de um polígono (prevalece o último do arquivo).')

else:
    with rasterio.open(ARQ_ENTRADA) as r:
        nodata_entrada = r.nodata
        mesma_grade = r.crs == GRADE_CRS and r.transform == GRADE_TRANSFORM and (r.height, r.width) == GRADE_SHAPE
        if mesma_grade:
            bruto = r.read(1).astype('int64')
        else:
            print('Aviso: raster de amostras fora da grade da imagem; reamostrado por vizinho mais próximo.')
            bruto = np.zeros(GRADE_SHAPE, dtype='int64')
            reproject(rasterio.band(r, 1), bruto, dst_transform=GRADE_TRANSFORM, dst_crs=GRADE_CRS,
                      src_nodata=nodata_entrada if nodata_entrada is not None else 0, dst_nodata=0,
                      resampling=Resampling.nearest)
    ausente = (bruto == 0) | (bruto == -9999)
    if nodata_entrada is not None:
        ausente |= bruto == nodata_entrada
    invalidos = sorted(np.unique(bruto[~ausente & ((bruto < 1) | (bruto > len(CLASSES)))]).tolist())
    if invalidos:
        raise ValueError(f'Códigos fora de 1..{len(CLASSES)} no raster: {invalidos} — os códigos seguem a ordem de cfg["classes"].')
    raster_classes = np.where(ausente, 0, bruto).astype('uint8')

    regioes = features.shapes(raster_classes, mask=raster_classes > 0, connectivity=4, transform=GRADE_TRANSFORM)
    geometrias, nomes = [], []
    for geom, valor in regioes:
        geometrias.append(shape(geom))
        nomes.append(CLASSES[int(valor) - 1])
    poligonos = gpd.GeoDataFrame({COL: nomes}, geometry=geometrias, crs=GRADE_CRS)

perfil.update(count=1, dtype='uint8', nodata=0, compress='lzw')
with rasterio.open(ARQ_RASTER_AMOSTRAS, 'w', **perfil) as dst:
    dst.write(raster_classes, 1)

area_pixel = abs(GRADE_TRANSFORM.a * GRADE_TRANSFORM.e)
pixels_poligono = (poligonos.area / area_pixel).round().astype(int)
pixels_raster = pd.Series(raster_classes[raster_classes > 0]).map({v: k for k, v in CODIGO.items()}).value_counts()
resumo_amostras = pd.DataFrame({'n_poligonos': poligonos.groupby(COL).size(),
                                'n_pixels': pixels_raster,
                                'poligonos_<10px': poligonos[pixels_poligono < 10].groupby(COL).size()})
resumo_amostras = resumo_amostras.reindex(CLASSES).fillna(0).astype(int)
print(f'Origem: {ORIGEM} | {ARQ_ENTRADA}\nRaster de amostras na grade da imagem: {ARQ_RASTER_AMOSTRAS}')
resumo_amostras.T

## 2. Lags por polígono e polarização

In [ ]:
lags_poligonos = ac.sample_lags(img['imagem'], poligonos, COL, target_correlation=ALVO_CORRELACAO,
                                value_transform=TRANSFORMACAO, estimator=ESTIMADOR, inclusive=INCLUSIVO,
                                plot=PLOTAR_PERFIS)
lags_poligonos['band'] = lags_poligonos['band'].map(lambda b: BANDAS[b - 1])     # 1, 2 -> nome da polarização
lags_poligonos.head()

## 3. Lags por classe

In [ ]:
resumo_lags = ac.summarize_lags(lags_poligonos, iqr=USAR_IQR)
if LAGS_MANUAIS is None:
    lags = ac.lags_for_sampling(resumo_lags, statistic=ESTATISTICA_LAG)
else:
    lags = pd.DataFrame(LAGS_MANUAIS, index=['h', 'v']).T.astype(int)
    print('Usando LAGS_MANUAIS.')
lags = lags.reindex([c for c in CLASSES if c in lags.index])
display(resumo_lags.round(2))
lags.T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, d in zip(axes, ['lag_h', 'lag_v']):
    for b, g in lags_poligonos.groupby('band'):
        pos = {c: i for i, c in enumerate(lags.index)}
        x = g['class'].map(pos) + (0.15 if b == BANDAS[1] else -0.15)
        ax.scatter(x, g[d], s=14, alpha=0.6, label=b)
    ax.plot(range(len(lags)), lags[d[-1]], 'k_', ms=18, mew=2, label='lag final')
    ax.set_xticks(range(len(lags)), lags.index); ax.set_title({'lag_h': 'Horizontal', 'lag_v': 'Vertical'}[d])
    ax.set_ylabel('Lag (pixels)'); ax.grid(axis='y', color='0.85')
axes[0].legend()
plt.suptitle(f'Lags por polígono e lag final por classe — {DATA}'); plt.tight_layout()
if pdf: pdf.savefig(fig)
plt.show()

## 4. Amostragem sistemática por lags

Para cada classe, o treino é formado pelos pixels da grade com passo `h × v`. A validação recebe os demais pixels da classe. A grade é aplicada sobre `amostras_raster.tif` (seção 1).

> A validação inclui vizinhos imediatos dos pixels de treino, portanto correlacionados com eles. Use-a com essa ressalva.

In [ ]:
treino, validacao = sampling.lag_grid_split(ARQ_RASTER_AMOSTRAS, CLASSES, lags)
contagem = pd.DataFrame({'treino': treino['class_name'].value_counts(), 'validacao': validacao['class_name'].value_counts()})
contagem = contagem.reindex(lags.index).fillna(0).astype(int)
contagem['total'] = contagem.sum(axis=1)
contagem['proporcao_treino_%'] = (100 * contagem['treino'] / contagem['total']).round(2)
contagem                                                                    # equivalente à Tabela 5.2

## 5. Gravação

In [ ]:
sampling.save_points(treino, ARQ_AMOSTRAS, layer='treino')
sampling.save_points(validacao, ARQ_AMOSTRAS, layer='validacao')
reporting.save_workbook(ARQ_PLANILHA, {
    'amostras_entrada': resumo_amostras.reset_index().rename(columns={'index': 'classe'}),
    'lags_poligonos': lags_poligonos,
    'resumo_por_classe_banda': resumo_lags,
    'lags_finais': lags.reset_index().rename(columns={'index': 'classe', 'class': 'classe'}),
    'amostras': contagem.reset_index().rename(columns={'index': 'classe'}),
    'parametros': pd.DataFrame({'parametro': ['DATA', 'AMOSTRAS', 'ORIGEM_AMOSTRAS', 'ALVO_CORRELACAO', 'ESTIMADOR',
                                              'INCLUSIVO', 'TRANSFORMACAO', 'USAR_IQR', 'ESTATISTICA_LAG', 'LAGS_MANUAIS'],
                                'valor': [DATA, ARQ_ENTRADA, ORIGEM, ALVO_CORRELACAO, ESTIMADOR, INCLUSIVO,
                                          TRANSFORMACAO, USAR_IQR, ESTATISTICA_LAG, str(LAGS_MANUAIS)]})})
if pdf: pdf.close()
print(f'Amostras: {ARQ_AMOSTRAS} (treino: {len(treino)} | validação: {len(validacao)})')